In [8]:
import duckdb
cn = duckdb.connect()
import os

sql = f"""
    CREATE OR REPLACE SECRET gcs_secrets (TYPE GCS, KEY_ID '{os.getenv("GCS_KEY")}', SECRET '{os.getenv("GCS_SECRET")}');
    
"""
cn.sql(sql)

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [12]:
cn.execute(
    f"ATTACH OR REPLACE 'ducklake:postgres:dbname=ducklake_v1 host=localhost' AS gcs_wh "
    f"(DATA_PATH 'gs://{os.getenv('GCS_BUCKET')}/ducklake')"
)

In [6]:
cn.sql("create table gcs_wh.orders (order_id integer, order_date date, customer_id integer, amount double)")
cn.sql("insert into gcs_wh.orders values (1, '2024-01-01', 1, 100.0), (2, '2024-01-02', 2, 150.0), (3, '2024-01-03', 1, 200.0)")
result = cn.sql("select * from gcs_wh.orders").fetchall()
print(result)

[(1, datetime.date(2024, 1, 1), 1, 100.0), (2, datetime.date(2024, 1, 2), 2, 150.0), (3, datetime.date(2024, 1, 3), 1, 200.0)]


In [13]:
cn.close()